In [1]:
!pip install pandas numpy scikit-learn scipy statsmodels xgboost torch -q

# Financial ML Pipeline — Tick Data, Expanded Features, Model Comparison

This notebook builds an end-to-end financial machine learning pipeline on real trade-by-trade tick data (Kibot format: `Date, Time, Price, Bid, Ask, Size`, no header).

Pipeline stages:
1. Ingestion and cleaning of raw tick data
2. Dollar bars (information-driven sampling)
3. Fractional differentiation (achieve stationarity while preserving memory)
4. Triple-barrier labeling with a CUSUM event filter
5. Feature engineering — 25 features across microstructure, entropy, and price/momentum families
6. Feature importance and selection (MDI)
7. Four classifiers: Logistic Regression, Random Forest, XGBoost, and an LSTM sequence model
8. Training and evaluation with purged K-fold cross-validation
9. Per-event realized-return statistics for each model's signal
10. Final summary dashboard comparing all four models

Every number in the final dashboard is computed from the actual data run through this notebook — there are no placeholder or synthetic values.

In [2]:
# ════════════════════════════════════════════════════════════════════════
# IMPORTS
# ════════════════════════════════════════════════════════════════════════
import time
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Device: {DEVICE}")

Device: cuda


---
## Configuration

This is the single place to point the notebook at the tick data file and tune the pipeline parameters.

- `DATA_PATH`: a Kibot-format tick file (`Date, Time, Price, Bid, Ask, Size`, no header).
- `DRY_RUN`: tick files can run into the millions of rows, so this toggle lets the whole pipeline be checked on a small recent slice before committing to a full run.
- `dollar_bar_usd = None`: the dollar-bar threshold is auto-calibrated in Stage 2 from the total traded dollar volume in the slice being used.

In [3]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG — EDIT THESE FOR THE DATA FILE
# ════════════════════════════════════════════════════════════════════════
DATA_PATH = "IVE_tickbidask.txt"   # <-- set this

DRY_RUN        = False       # set False to run on the full file
DRY_RUN_MODE   = "rows"     # "rows" -> last N rows, "dates" -> a date range
DRY_RUN_N_ROWS = 50_000
DRY_RUN_START  = "2009-09-28"
DRY_RUN_END    = "2009-10-05"

CFG = dict(
    # Bars
    dollar_bar_usd = None,          # auto-calibrated in Stage 2
    target_bars_per_day = 50,
    trading_days = 252,

    # Fractional differentiation
    ffd_thres = 1e-4,
    adf_pval  = 0.05,

    # Triple-barrier labeling
    cusum_h_mult = 1.0,      # h = cusum_h_mult * daily_vol
    pt_sl = [1.0, 1.0],      # profit-take / stop-loss multiples of sigma
    vert_bars = 5,           # vertical barrier, in bars
    min_ret = 0.0,

    # Feature engineering
    micro_window = 15,
    long_window = 60,

    # Feature selection
    n_top_features = 15,

    # Sequence model (LSTM only)
    seq_len = 20,
    lstm_hidden = 48,
    lstm_layers = 1,
    n_classes = 2,
    n_epochs = 15,
    batch_size = 64,
    lr = 1e-3,

    # Training
    n_folds = 5,
    pct_embargo = 0.01,

    # Cost assumption for the return stats
    cost_bps = 1.0,          # extra cost on top of the real quoted spread, in bps
)
print("Config loaded.")

Config loaded.


---
## Stage 1 — Ingestion and Cleaning

Real data arrives as trade-by-trade ticks with a contemporaneous best bid/ask quote and traded size on every row. This stage:

1. Parses the raw Kibot-format CSV, builds a proper datetime index, and derives `v` (traded volume) and `dv` (traded dollar value) per tick.
2. Optionally restricts to a dry-run slice so the rest of the pipeline can be validated quickly.
3. Drops invalid rows: non-positive price/bid/ask, or crossed quotes (`ask < bid`).
4. Runs a MAD (median absolute deviation) outlier filter on price.

In [4]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 1 — Ingestion and Cleaning
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 1 — Ingestion and Cleaning ".center(65, "="))
print("=" * 65)

def read_kibot_ticks(fp: str) -> pd.DataFrame:
    cols = ['date', 'time', 'price', 'bid', 'ask', 'size']
    df = pd.read_csv(fp, header=None, names=cols)
    df['dates'] = pd.to_datetime(df['date'] + df['time'], format='%m/%d/%Y%H:%M:%S')
    df['v'] = df['size']
    df['dv'] = df['price'] * df['size']
    df = df.drop(columns=['date', 'time']).set_index('dates').drop_duplicates()
    return df.sort_index()

def mad_filter(s: pd.Series, n_mad: float = 8.0) -> pd.Series:
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0:
        return s
    z = 0.6745 * (s - med) / mad
    return s[z.abs() <= n_mad]

t0 = time.time()
df_all = read_kibot_ticks(DATA_PATH)
print(f"Loaded {len(df_all):,} raw ticks in {time.time() - t0:.1f}s")

if DRY_RUN:
    if DRY_RUN_MODE == "rows":
        df_all = df_all.iloc[-DRY_RUN_N_ROWS:]
    else:
        df_all = df_all.loc[DRY_RUN_START:DRY_RUN_END]
    print(f"DRY_RUN active ({DRY_RUN_MODE}) -> {len(df_all):,} rows")

valid = (df_all['price'] > 0) & (df_all['bid'] > 0) & (df_all['ask'] > 0) & (df_all['ask'] >= df_all['bid'])
df_all = df_all[valid]

keep_idx = mad_filter(df_all['price']).index
TRADES = df_all.loc[keep_idx].copy()
TRADES['spread'] = TRADES['ask'] - TRADES['bid']

n_days_span = TRADES.index.normalize().nunique()
print(f"Clean ticks: {len(TRADES):,}  |  Trading days spanned: {n_days_span}")
TRADES.head()


================ STAGE 1 — Ingestion and Cleaning ===============
Loaded 3,313,815 raw ticks in 58.5s
Clean ticks: 10,702,026  |  Trading days spanned: 4213


,price,bid,ask,size,v,dv,spread
dates,,,,,,,
2009-09-28 09:30:00,50.79,50.70,50.79,100,100,5079.00,0.09
2009-09-28 09:30:00,50.71,50.70,50.79,638,638,32352.98,0.09
2009-09-28 09:30:00,50.79,50.70,50.79,100,100,5079.00,0.09
2009-09-28 09:30:00,50.71,50.70,50.79,638,638,32352.98,0.09
2009-09-28 09:31:32,50.75,50.75,50.76,100,100,5075.00,0.01


---
## Stage 2 — Dollar Bars

Rather than sampling at fixed time intervals, which oversamples during quiet periods and undersamples during bursts of activity, a new bar closes every time cumulative traded dollar value crosses a threshold. This gives bars with more uniform statistical properties than time bars.

The threshold is auto-calibrated: total traded dollar volume in the slice is divided by a target bar count, so the resulting bar frequency scales with however much data is being used. Each bar also carries a Lee-Ready style trade sign (tick rule) and the average quoted spread over the bar, which feed into the feature stage later.

The bar index is de-duplicated (keeping the later bar) right after construction, since bars can end on ticks sharing the same timestamp and every later stage relies on this index being unique.

In [5]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 2 — Dollar Bars
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 2 — Dollar Bars ".center(65, "="))
print("=" * 65)

total_dv = TRADES['dv'].sum()
n_target_bars = CFG['target_bars_per_day'] * max(n_days_span, 1)
if CFG['dollar_bar_usd'] is None:
    CFG['dollar_bar_usd'] = total_dv / max(n_target_bars, 1)
print(f"Dollar-bar threshold: {CFG['dollar_bar_usd']:,.0f}")

def tick_rule_signs(price: pd.Series) -> np.ndarray:
    diff = price.diff().fillna(0.0)
    signs = np.sign(diff)
    last = 1.0
    out = np.zeros(len(price))
    for i, s in enumerate(signs.values):
        if s == 0:
            out[i] = last
        else:
            out[i] = s
            last = s
    return out

TRADES['sign'] = tick_rule_signs(TRADES['price'])

def build_dollar_bars(trades: pd.DataFrame, thresh: float) -> pd.DataFrame:
    bars = []
    buf = []
    cum_dv = 0.0
    for ts, row in trades.iterrows():
        buf.append(row)
        cum_dv += row['dv']
        if cum_dv >= thresh:
            prices = [r['price'] for r in buf]
            vols = [r['v'] for r in buf]
            dvols = [r['dv'] for r in buf]
            signs = [r['sign'] for r in buf]
            spreads = [r['spread'] for r in buf]
            buy_v = sum(v for v, s in zip(vols, signs) if s > 0)
            sell_v = sum(v for v, s in zip(vols, signs) if s <= 0)
            bars.append(dict(
                dates=ts, open=prices[0], high=max(prices), low=min(prices), close=prices[-1],
                volume=sum(vols), n_ticks=len(buf), dvol=sum(dvols),
                vwap=sum(dvols) / max(sum(vols), 1),
                buy_vol=buy_v, sell_vol=sell_v,
                spread=float(np.mean(spreads)),
            ))
            buf = []
            cum_dv = 0.0
    out = pd.DataFrame(bars).set_index('dates')
    return out

t0 = time.time()
BARS = build_dollar_bars(TRADES, CFG['dollar_bar_usd'])
BARS = BARS[~BARS.index.duplicated(keep='last')].sort_index()
print(f"Built {len(BARS):,} dollar bars in {time.time() - t0:.1f}s")
BARS.head()


===================== STAGE 2 — Dollar Bars =====================
Dollar-bar threshold: 3,002,353
Built 103,303 dollar bars in 414.0s


,open,high,low,close,volume,n_ticks,dvol,vwap,buy_vol,sell_vol,spread
dates,,,,,,,,,,,
2009-09-28 09:46:35,50.79,51.07,50.710,51.0700,58976.0,156,3.006168e+06,50.972743,35562.0,23414.0,0.020577
2009-09-28 09:53:49,51.07,51.15,51.050,51.1400,59626.0,110,3.046581e+06,51.094838,50876.0,8750.0,0.015091
2009-09-28 09:54:41,51.14,51.14,51.129,51.1400,59053.0,104,3.019619e+06,51.134043,44753.0,14300.0,0.011635
2009-09-28 10:01:06,51.14,51.26,51.080,51.2301,59644.0,182,3.051822e+06,51.167292,37785.0,21859.0,0.019231
2009-09-28 10:02:52,51.25,51.26,51.250,51.2500,59500.0,67,3.049506e+06,51.252208,12600.0,46900.0,0.015522


---
## Stage 3 — Fractional Differentiation

Integer differencing (returns) achieves stationarity but destroys memory. Fractional differencing generalizes to a non-integer order *d*:

```
(1-L)^d X_t = sum_{k=0}^inf C(d,k)(-1)^k X_{t-k}
```

Weights: `w_0 = 1`, `w_k = -w_{k-1} * (d-k+1)/k`, decaying as `O(k^{-d-1})` — distant lags still contribute, so the series keeps long memory.

**Fixed-Width Window (FFD)**: truncate weights below a threshold for efficiency.
**min-d\***: binary search for the smallest *d* such that the ADF test rejects a unit root at the 5% level.

The resulting fractionally-differenced series is carried into Stage 5 as an actual feature, along with its first difference — it is not just a diagnostic.

In [6]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 3 — Fractional Differentiation
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 3 — Fractional Differentiation ".center(65, "="))
print("=" * 65)

def ffd_weights(d: float, thres: float = 1e-4) -> np.ndarray:
    w = [1.0]; k = 1
    while True:
        wk = -w[-1] * (d - k + 1) / k
        if abs(wk) < thres:
            break
        w.append(wk); k += 1
    return np.array(w[::-1], dtype=np.float64)

def frac_diff_ffd(series: pd.Series, d: float, thres: float = 1e-4) -> pd.Series:
    vals = series.values.astype(np.float64)
    w = ffd_weights(d, thres); wlen = len(w); n = len(vals)
    if wlen > n:
        return pd.Series(dtype=float)
    out = np.full(n, np.nan)
    for i in range(wlen - 1, n):
        out[i] = np.dot(w, vals[i - wlen + 1:i + 1])
    return pd.Series(out, index=series.index).dropna()

def min_ffd_d(series: pd.Series, thres: float, pval: float = 0.05) -> float:
    for d in np.linspace(0, 1, 21):
        diffed = frac_diff_ffd(series, d, thres)
        if len(diffed) < 20:
            continue
        p = adfuller(diffed, maxlag=1, regression='c', autolag=None)[1]
        if p < pval:
            return float(d)
    return 1.0

t0 = time.time()
d_star = min_ffd_d(BARS['close'], CFG['ffd_thres'], CFG['adf_pval'])
FFD_CLOSE = frac_diff_ffd(BARS['close'], d_star, CFG['ffd_thres'])
print(f"min-d* = {d_star:.4f}  |  FFD series length: {len(FFD_CLOSE):,}  ({time.time() - t0:.1f}s)")
FFD_CLOSE.head()


============== STAGE 3 — Fractional Differentiation =============
min-d* = 0.2000  |  FFD series length: 102,807  (1.6s)


,0
dates,
2009-12-09 09:43:48,12.805841
2009-12-09 12:16:20,13.014219
2009-12-09 13:16:35,12.965176
2009-12-09 15:49:42,13.273173
2009-12-10 09:52:32,13.562069


---
## Stage 4 — Triple-Barrier Labeling

1. **CUSUM filter**: only sample event times at structurally significant price moves (a symmetric positive/negative cumulative-sum filter with threshold `h`), which reduces noise relative to sampling every bar.
2. **Triple-barrier method**: for each event `t0`, set three barriers —
   - Upper (profit-take): price rises by `pt_sl[0] * sigma_t0` -> label = 1
   - Lower (stop-loss): price falls by `pt_sl[1] * sigma_t0` -> label = 0
   - Vertical (time): price at `t0 + vert_bars` -> label = sign of the realized return
   The label is whichever barrier is touched first.
3. **Sample uniqueness weights**: each label's weight is inversely proportional to how much its `[t0, t1]` window overlaps with other labels' windows, so overlapping/redundant events count less during training.

In [7]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 4 — Triple-Barrier Labeling
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 4 — Triple-Barrier Labeling ".center(65, "="))
print("=" * 65)

def daily_vol(close: pd.Series, span: int = 50) -> pd.Series:
    r = close.pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    return r.ewm(span=span, min_periods=span).std().reindex(close.index).ffill()

def cusum_filter(g: pd.Series, h: float) -> pd.DatetimeIndex:
    s_pos = s_neg = 0.0
    dg = g.diff().dropna()
    events = []
    for t, v in dg.items():
        s_pos = max(0.0, s_pos + v); s_neg = min(0.0, s_neg + v)
        if s_pos >= h:
            s_pos = 0.0; events.append(t)
        elif s_neg <= -h:
            s_neg = 0.0; events.append(t)
    return pd.DatetimeIndex(events)

def triple_barrier(close: pd.Series, t_events: pd.DatetimeIndex, sigma: pd.Series,
                    pt_sl: list, vert_bars: int, min_ret: float = 0.0) -> pd.DataFrame:
    idx = close.index
    pos = {t: i for i, t in enumerate(idx)}
    out = []
    for t0 in t_events:
        if t0 not in pos:
            continue
        i0 = pos[t0]
        i1 = min(i0 + vert_bars, len(idx) - 1)
        if i1 <= i0:
            continue
        s = sigma.get(t0, np.nan)
        if pd.isna(s) or s < min_ret:
            continue
        path = close.iloc[i0:i1 + 1]
        ret_path = path / close.iloc[i0] - 1.0
        upper = pt_sl[0] * s
        lower = -pt_sl[1] * s
        hit_up = ret_path[ret_path >= upper]
        hit_dn = ret_path[ret_path <= lower]
        t_up = hit_up.index[0] if len(hit_up) else pd.NaT
        t_dn = hit_dn.index[0] if len(hit_dn) else pd.NaT
        t_vert = idx[i1]
        candidates = [t for t in [t_up, t_dn, t_vert] if pd.notna(t)]
        t1 = min(candidates)
        ret = close.loc[t1] / close.loc[t0] - 1.0
        label = 1 if ret > 0 else 0
        out.append(dict(t0=t0, t1=t1, ret=ret, label=label))
    return pd.DataFrame(out).set_index('t0')

def uniqueness_weights(events: pd.DataFrame, idx: pd.Index) -> pd.Series:
    pos = {t: i for i, t in enumerate(idx)}
    n = len(idx)
    concurrency = np.zeros(n)
    for t0, row in events.iterrows():
        i0, i1 = pos[t0], pos.get(row['t1'], n - 1)
        concurrency[i0:i1 + 1] += 1
    weights = []
    for t0, row in events.iterrows():
        i0, i1 = pos[t0], pos.get(row['t1'], n - 1)
        c = concurrency[i0:i1 + 1]
        weights.append(1.0 / c.mean() if c.mean() > 0 else 1.0)
    w = pd.Series(weights, index=events.index)
    return w * len(w) / w.sum()

SIGMA = daily_vol(BARS['close'])
h = CFG['cusum_h_mult'] * SIGMA.mean()
t_events = cusum_filter(BARS['close'], h)
EVENTS = triple_barrier(BARS['close'], t_events, SIGMA, CFG['pt_sl'], CFG['vert_bars'], CFG['min_ret'])
EVENTS['w'] = uniqueness_weights(EVENTS, BARS.index)

print(f"CUSUM events: {len(t_events):,} -> {len(EVENTS):,} valid labeled events")
print(f"Label balance: {EVENTS['label'].value_counts(normalize=True).round(3).to_dict()}")
EVENTS.head()


=============== STAGE 4 — Triple-Barrier Labeling ===============
CUSUM events: 99,046 -> 98,999 valid labeled events
Label balance: {1: 0.515, 0: 0.485}


,t1,ret,label,w
t0,,,,
2009-10-01 10:02:20,2009-10-01 10:06:06,-0.003159,0,2.689150
2009-10-01 10:06:06,2009-10-01 11:41:40,-0.003401,0,1.440616
2009-10-01 10:23:02,2009-10-01 11:23:02,-0.003639,0,1.344575
2009-10-01 10:55:23,2009-10-01 11:23:02,-0.004426,0,1.152493
2009-10-01 11:23:02,2009-10-01 13:09:57,-0.003890,0,1.210117


---
## Stage 5 — Feature Engineering (25 Features)

This dataset carries genuine bid/ask/size on every trade, so a richer feature set is built across four families:

**Fractional differentiation** (2): the stationary FFD close series from Stage 3, and its first difference.

**Microstructure** (9): relative spread and its rolling z-score, Amihud illiquidity, a Kyle's-lambda-style price-impact measure, VWAP deviation, buy-volume ratio, tick-count z-score, volume z-score, dollar-volume z-score.

**Entropy / information** (2): rolling Shannon entropy of binned returns, and its change.

**Price / momentum / distribution** (12): short and long rolling returns, short and long rolling volatility, two moving-average ratios at different horizons, an RSI-style oscillator, rolling skewness, rolling kurtosis, rolling lag-1 autocorrelation, high-low range, and the intrabar open-to-close return.

In [8]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 5 — Feature Engineering (25 features)
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 5 — Feature Engineering ".center(65, "="))
print("=" * 65)

w = CFG['micro_window']
lw = CFG['long_window']

def zscore(s, window):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

def amihud_illiq(close, dvol, window):
    illiq = close.pct_change().abs() / dvol.replace(0, np.nan)
    return illiq.rolling(window).mean()

def kyle_lambda(close, volume, spread, window):
    lam = close.diff().abs() / volume.replace(0, np.nan)
    return lam.rolling(window).mean() * (1 + spread.rolling(window).mean().fillna(0))

def rolling_entropy(returns, window, n_bins=8):
    def _ent(x):
        if np.all(np.isnan(x)):
            return np.nan
        counts, _ = np.histogram(x, bins=n_bins)
        p = counts / counts.sum()
        p = p[p > 0]
        return float(-(p * np.log(p)).sum())
    return returns.rolling(window).apply(_ent, raw=True)

def rsi(close, window):
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(window).mean()
    loss = (-delta.clip(upper=0)).rolling(window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def rolling_autocorr(returns, window, lag=1):
    return returns.rolling(window).apply(lambda x: pd.Series(x).autocorr(lag=lag), raw=False)

close, opn, high, low = BARS['close'], BARS['open'], BARS['high'], BARS['low']
ret = close.pct_change()

FEATURES = pd.DataFrame(index=BARS.index)

# fractional differentiation
FEATURES['ffd_close'] = FFD_CLOSE.reindex(BARS.index)
FEATURES['ffd_close_chg'] = FEATURES['ffd_close'].diff()

# microstructure
FEATURES['rel_spread'] = (BARS['spread'] / close).rolling(w).mean()
FEATURES['rel_spread_z'] = zscore(BARS['spread'] / close, w)
FEATURES['amihud'] = amihud_illiq(close, BARS['dvol'], w)
FEATURES['kyle_lambda'] = kyle_lambda(close, BARS['volume'], BARS['spread'] / close, w)
FEATURES['vwap_dev'] = (close - BARS['vwap']) / close
FEATURES['buy_ratio'] = BARS['buy_vol'] / (BARS['buy_vol'] + BARS['sell_vol']).replace(0, np.nan)
FEATURES['n_ticks_z'] = zscore(BARS['n_ticks'], w)
FEATURES['volume_z'] = zscore(BARS['volume'], w)
FEATURES['dvol_z'] = zscore(BARS['dvol'], w)

# entropy / information
FEATURES['entropy'] = rolling_entropy(ret, w)
FEATURES['entropy_chg'] = FEATURES['entropy'].diff()

# price / momentum / distribution
FEATURES['ret_mom_short'] = close.pct_change(w)
FEATURES['ret_mom_long'] = close.pct_change(lw)
FEATURES['volatility_short'] = ret.rolling(w).std()
FEATURES['volatility_long'] = ret.rolling(lw).std()
FEATURES['ma_ratio_short'] = close.rolling(w).mean() / close.rolling(w * 2).mean() - 1.0
FEATURES['ma_ratio_long'] = close.rolling(w * 2).mean() / close.rolling(lw).mean() - 1.0
FEATURES['rsi'] = rsi(close, w)
FEATURES['skew'] = ret.rolling(lw).skew()
FEATURES['kurtosis'] = ret.rolling(lw).kurt()
FEATURES['autocorr_1'] = rolling_autocorr(ret, lw, lag=1)
FEATURES['hl_range'] = (high - low) / close
FEATURES['oc_ret'] = close / opn - 1.0

FEATURES = FEATURES.replace([np.inf, -np.inf], np.nan).dropna()
print(f"Features built: {FEATURES.shape[1]} columns, {len(FEATURES):,} rows")
assert FEATURES.shape[1] == 25, f"expected 25 features, got {FEATURES.shape[1]}"
FEATURES.head()


================= STAGE 5 — Feature Engineering =================
Features built: 25 columns, 102,804 rows


,ffd_close,ffd_close_chg,rel_spread,rel_spread_z,amihud,kyle_lambda,vwap_dev,buy_ratio,n_ticks_z,volume_z,...,volatility_short,volatility_long,ma_ratio_short,ma_ratio_long,rsi,skew,kurtosis,autocorr_1,hl_range,oc_ret
dates,,,,,,,,,,,,,,,,,,,,,
2009-12-09 12:16:20,13.014219,0.208377,0.000382,-0.347827,9.460028e-10,0.000003,-0.000575,0.518594,1.537701,-0.101520,...,0.003867,0.003910,-0.002851,-0.000301,32.248521,1.167652,5.696768,0.143864,0.006434,0.002738
2009-12-09 13:16:35,12.965176,-0.049042,0.000381,-0.181259,9.639369e-10,0.000003,0.000626,0.293225,0.004804,0.820969,...,0.003856,0.003898,-0.003158,-0.001186,31.632739,1.220674,5.882244,0.142649,0.002671,-0.000827
2009-12-09 15:49:42,13.273173,0.307996,0.000382,-0.357328,1.084088e-09,0.000003,0.003637,0.470624,0.823101,-0.510350,...,0.004221,0.003961,-0.003449,-0.001830,38.913942,1.129986,5.261188,0.141293,0.008539,0.005610
2009-12-10 09:52:32,13.562069,0.288897,0.000398,2.112557,1.208686e-09,0.000003,0.003973,0.388170,0.903063,-0.313203,...,0.004607,0.003973,-0.003667,-0.002082,46.386356,1.127705,5.179376,0.124243,0.010745,0.006605
2009-12-10 12:00:53,13.370618,-0.191452,0.000399,-0.377186,1.191106e-09,0.000003,-0.000733,0.409252,0.477654,-0.127981,...,0.004537,0.003805,-0.004153,-0.002217,41.755411,1.204403,6.418334,0.089110,0.004676,-0.002074


---
## Stage 6 — Feature Importance and Selection

Mean Decrease Impurity (MDI) from a bagged, class-balanced Random Forest fit on all labeled events, sample-weighted by uniqueness. This ranks the 25 raw features and keeps the top-K, which keeps the model set manageable and shows which of the microstructure, entropy, and momentum families actually carry signal for this instrument.

In [9]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 6 — Feature Importance and Selection (MDI)
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 6 — Feature Importance and Selection ".center(65, "="))
print("=" * 65)

common_idx = FEATURES.index.intersection(EVENTS.index)
X_fs = FEATURES.loc[common_idx]
y_fs = EVENTS.loc[common_idx, 'label']
w_fs = EVENTS.loc[common_idx, 'w']

rf_fs = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                                random_state=42, n_jobs=-1)
rf_fs.fit(X_fs, y_fs, sample_weight=w_fs.values)
mdi = pd.Series(rf_fs.feature_importances_, index=X_fs.columns).sort_values(ascending=False)

n_top = min(CFG['n_top_features'], len(mdi))
top_feats = mdi.index[:n_top].tolist()
print(f"Top {n_top} of {len(mdi)} features by MDI:")
print(mdi.head(n_top).round(4))


=========== STAGE 6 — Feature Importance and Selection ==========
Top 15 of 25 features by MDI:
ffd_close           0.0788
ret_mom_long        0.0616
kurtosis            0.0569
ret_mom_short       0.0563
skew                0.0541
volatility_long     0.0540
rel_spread          0.0514
rsi                 0.0471
ma_ratio_long       0.0438
autocorr_1          0.0417
kyle_lambda         0.0400
oc_ret              0.0385
ma_ratio_short      0.0375
volatility_short    0.0354
rel_spread_z        0.0343
dtype: float64


---
## Stage 7 — Four Classifiers: Logistic Regression, Random Forest, XGBoost, LSTM

Four models of increasing complexity are trained on the same top selected features and compared on identical purged folds:

- **Logistic Regression**: a simple linear baseline on standardized features.
- **Random Forest**: a bagged ensemble of decision trees, robust to non-linear relationships.
- **XGBoost**: a gradient-boosted tree ensemble — typically the strongest tabular model, and standard in quant research.
- **LSTM**: a recurrent network over a window of `seq_len` bars of the same features, to check whether modeling short-term sequence structure adds anything the tabular models miss.

```
LSTM — Input: (B, L, F) sequence of F-dim feature vectors, length L (seq_len)
LSTM -> final hidden state h_T -> Linear(hidden -> n_classes) -> logits
```

In [10]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 7 — Model Definitions
# ════════════════════════════════════════════════════════════════════════
def make_logreg():
    return LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)

def make_rf():
    return RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                                   random_state=42, n_jobs=-1)

def make_xgb():
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
                          random_state=42, n_jobs=-1)

class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, w: np.ndarray, seq_len: int):
        self.X, self.y, self.w, self.seq_len = X, y, w, seq_len
        self.valid_starts = np.arange(seq_len - 1, len(X))

    def __len__(self):
        return len(self.valid_starts)

    def __getitem__(self, i):
        end = self.valid_starts[i]
        start = end - self.seq_len + 1
        seq = self.X[start:end + 1]
        return (torch.tensor(seq, dtype=torch.float32),
                torch.tensor(self.y[end], dtype=torch.long),
                torch.tensor(self.w[end], dtype=torch.float32))

class LSTMClassifier(nn.Module):
    def __init__(self, feature_dim, hidden=48, n_layers=1, n_classes=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(feature_dim, hidden, num_layers=n_layers,
                             batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, n_classes))

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])

def make_lstm(f_dim):
    return LSTMClassifier(feature_dim=f_dim, hidden=CFG['lstm_hidden'],
                           n_layers=CFG['lstm_layers'], n_classes=CFG['n_classes']).to(DEVICE)

def train_lstm_epoch(model, loader, optim):
    model.train(); total = 0.0
    for xb, yb, wb in loader:
        xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
        optim.zero_grad()
        logits = model(xb)
        loss = (F.cross_entropy(logits, yb, reduction='none') * wb).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        total += loss.item() * len(yb)
    return total / len(loader.dataset)

@torch.no_grad()
def predict_lstm(model, loader):
    model.eval()
    preds = []
    for xb, _, _ in loader:
        logits = model(xb.to(DEVICE))
        preds.append(logits.argmax(-1).cpu().numpy())
    return np.concatenate(preds)

print("Logistic Regression, Random Forest, XGBoost, and LSTM models defined.")

Logistic Regression, Random Forest, XGBoost, and LSTM models defined.


---
## Stage 8 — Training and Evaluation with Purged K-Fold Cross-Validation

Standard K-Fold leaks whenever a label's outcome window `[t0, t1]` crosses a fold boundary. **PurgedKFold** fixes this by:

1. **Purging**: dropping training observations whose `t1 >= test_start` (their label would otherwise bleed information into the test set).
2. **Embargo**: dropping training observations in the first `pct_embargo` fraction of bars after the test window (a buffer against serial autocorrelation).

Folds are walk-forward (strictly time-ordered, no shuffling). All four models are trained and evaluated on the same folds and the same feature set (the LSTM additionally windows the features into `seq_len`-length sequences), so their out-of-sample results are directly comparable.

In [11]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 8 — Training and Evaluation with Purged K-Fold CV
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 8 — Training and Evaluation with Purged K-Fold CV ".center(65, "="))
print("=" * 65)

class PurgedKFold:
    def __init__(self, n_splits, t1: pd.Series, pct_embargo=0.01):
        self.n_splits, self.t1, self.pct_embargo = n_splits, t1, pct_embargo

    def split(self, idx):
        n = len(idx)
        emb = int(n * self.pct_embargo)
        fold_size = n // self.n_splits
        for fold in range(self.n_splits):
            t_start = fold * fold_size
            t_end = t_start + fold_size if fold < self.n_splits - 1 else n
            test_idx = np.arange(t_start, t_end)
            test_t0, test_t1 = idx[test_idx[0]], idx[test_idx[-1]]
            train_mask = np.ones(n, dtype=bool)
            train_mask[test_idx] = False
            for i, t0 in enumerate(idx):
                if not train_mask[i]:
                    continue
                t1_i = self.t1.get(t0, t0)
                if t0 <= test_t1 and t1_i >= idx[test_idx[0]]:
                    train_mask[i] = False
            embargo_end = min(t_end + emb, n)
            train_mask[t_end:embargo_end] = False
            train_idx = np.where(train_mask)[0]
            yield train_idx, test_idx

X_all = FEATURES.loc[common_idx, top_feats]
y_all = y_fs
w_all = w_fs
idx_all = common_idx

t1_map = EVENTS['t1']
pkf = PurgedKFold(n_splits=CFG['n_folds'], t1=t1_map, pct_embargo=CFG['pct_embargo'])

fold_metrics = []
model_names = ['logreg', 'rf', 'xgb', 'lstm']
oos_cols = [f"{m}_pred" for m in model_names]
oos_preds = pd.DataFrame(index=idx_all, columns=oos_cols, dtype=float)

for fold, (tr_idx, te_idx) in enumerate(pkf.split(idx_all)):
    if len(tr_idx) < 50 + CFG['seq_len'] or len(te_idx) < 10 + CFG['seq_len']:
        continue
    X_tr, X_te = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    y_tr, y_te = y_all.iloc[tr_idx], y_all.iloc[te_idx]
    w_tr = w_all.iloc[tr_idx]
    te_dates = idx_all[te_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    logreg = make_logreg()
    logreg.fit(X_tr_s, y_tr, sample_weight=w_tr.values)
    logreg_pred = logreg.predict(X_te_s)

    rf = make_rf()
    rf.fit(X_tr, y_tr, sample_weight=w_tr.values)
    rf_pred = rf.predict(X_te)

    xgb = make_xgb()
    xgb.fit(X_tr, y_tr, sample_weight=w_tr.values)
    xgb_pred = xgb.predict(X_te)

    # LSTM: build sequences within this fold only (train and test are already time-separated by the split)
    lstm_train_ds = SequenceDataset(X_tr_s, y_tr.values, w_tr.values, CFG['seq_len'])
    lstm_test_ds = SequenceDataset(X_te_s, y_te.values, np.ones(len(y_te)), CFG['seq_len'])
    train_loader = DataLoader(lstm_train_ds, batch_size=CFG['batch_size'], shuffle=True)
    test_loader = DataLoader(lstm_test_ds, batch_size=CFG['batch_size'], shuffle=False)

    lstm_model = make_lstm(len(top_feats))
    optim = torch.optim.AdamW(lstm_model.parameters(), lr=CFG['lr'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=CFG['n_epochs'])
    for _ in range(CFG['n_epochs']):
        train_lstm_epoch(lstm_model, train_loader, optim)
        sched.step()
    lstm_pred_seq = predict_lstm(lstm_model, test_loader)
    lstm_te_dates = te_dates[CFG['seq_len'] - 1:]

    preds = dict(logreg=logreg_pred, rf=rf_pred, xgb=xgb_pred)
    accs, f1s = {}, {}
    for name, pred in preds.items():
        oos_preds.loc[te_dates, f"{name}_pred"] = pred
        accs[name] = accuracy_score(y_te, pred)
        f1s[name] = f1_score(y_te, pred, zero_division=0)

    oos_preds.loc[lstm_te_dates, 'lstm_pred'] = lstm_pred_seq
    y_te_lstm = y_te.loc[lstm_te_dates]
    accs['lstm'] = accuracy_score(y_te_lstm, lstm_pred_seq)
    f1s['lstm'] = f1_score(y_te_lstm, lstm_pred_seq, zero_division=0)

    row = dict(fold=fold, n_test=len(te_idx))
    for name in model_names:
        row[f"{name}_acc"] = accs[name]
        row[f"{name}_f1"] = f1s[name]
    fold_metrics.append(row)

    print(f"Fold {fold}: " + "  |  ".join(f"{name.upper()} acc={accs[name]:.4f} f1={f1s[name]:.4f}" for name in model_names))

FOLD_METRICS = pd.DataFrame(fold_metrics)
oos_preds = oos_preds.dropna()
print()
for name in model_names:
    print(f"Mean OOS — {name.upper():8s}: acc={FOLD_METRICS[f'{name}_acc'].mean():.4f}  f1={FOLD_METRICS[f'{name}_f1'].mean():.4f}")


==== STAGE 8 — Training and Evaluation with Purged K-Fold CV ====
Fold 0: LOGREG acc=0.5213 f1=0.6117  |  RF acc=0.5192 f1=0.6654  |  XGB acc=0.5176 f1=0.6546  |  LSTM acc=0.5126 f1=0.6226
Fold 1: LOGREG acc=0.5184 f1=0.6232  |  RF acc=0.5020 f1=0.5651  |  XGB acc=0.5041 f1=0.5899  |  LSTM acc=0.5022 f1=0.5537
Fold 2: LOGREG acc=0.5106 f1=0.5253  |  RF acc=0.5046 f1=0.5114  |  XGB acc=0.5084 f1=0.5709  |  LSTM acc=0.5074 f1=0.5461
Fold 3: LOGREG acc=0.5098 f1=0.4425  |  RF acc=0.5092 f1=0.5142  |  XGB acc=0.5026 f1=0.5740  |  LSTM acc=0.4930 f1=0.5078
Fold 4: LOGREG acc=0.4985 f1=0.2928  |  RF acc=0.4944 f1=0.0343  |  XGB acc=0.4933 f1=0.1918  |  LSTM acc=0.5063 f1=0.4949

Mean OOS — LOGREG  : acc=0.5117  f1=0.4991
Mean OOS — RF      : acc=0.5059  f1=0.4581
Mean OOS — XGB     : acc=0.5052  f1=0.5162
Mean OOS — LSTM    : acc=0.5043  f1=0.5450


---
## Stage 9 — Per-Event Realized-Return Statistics

Rather than building a compounded equity curve over many overlapping event windows (which distorts the result, since the same period of time gets compounded through multiple overlapping labels), this stage reports simple, non-compounded statistics on the realized return of every out-of-sample event where a model's predicted label was followed:

- Mean and standard deviation of realized returns, net of the real quoted spread plus a small extra cost buffer.
- Hit rate: the fraction of followed signals that were net profitable.
- A Sharpe-like ratio: `mean / std`, annualized by the average number of such events per year — a measure of risk-adjusted edge per event, without any compounding assumption.

This is computed for all four models so they can be compared on the same basis as their classification metrics.

In [12]:
# ════════════════════════════════════════════════════════════════════════
# STAGE 9 — Per-Event Realized-Return Statistics
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" STAGE 9 — Per-Event Realized-Return Statistics ".center(65, "="))
print("=" * 65)

def signal_return_stats(pred_label, realized_ret, spread_frac, extra_cost_bps, n_days_span, name):
    direction = np.where(pred_label == 1, 1.0, -1.0)
    cost = spread_frac.fillna(extra_cost_bps / 1e4) + extra_cost_bps / 1e4
    net_ret = direction * realized_ret - cost
    net_ret = net_ret.dropna()

    n = len(net_ret)
    mean_ret = net_ret.mean()
    std_ret = net_ret.std()
    hit_rate = (net_ret > 0).mean()
    events_per_year = n / max(n_days_span / 252.0, 1e-6)
    sharpe_like = (mean_ret / std_ret) * np.sqrt(events_per_year) if std_ret > 0 else 0.0

    print(f"\n{name}")
    print(f"  Events followed:        {n:,}")
    print(f"  Mean net return/event:  {mean_ret:.5f}")
    print(f"  Std net return/event:   {std_ret:.5f}")
    print(f"  Hit rate:               {hit_rate:.3%}")
    print(f"  Sharpe-like ratio:      {sharpe_like:.3f}")
    return dict(name=name, n=n, mean_ret=mean_ret, std_ret=std_ret, hit_rate=hit_rate, sharpe_like=sharpe_like)

STATS = {}
for name in model_names:
    col = f"{name}_pred"
    valid = oos_preds[col].dropna()
    realized_ret = EVENTS.loc[valid.index, 'ret']
    spread_frac = BARS.loc[valid.index, 'spread'] / BARS.loc[valid.index, 'close']
    STATS[name] = signal_return_stats(valid.values, realized_ret, spread_frac,
                                       CFG['cost_bps'], n_days_span, f"{name.upper()} signal")


========= STAGE 9 — Per-Event Realized-Return Statistics ========

LOGREG signal
  Events followed:        98,468
  Mean net return/event:  -0.00037
  Std net return/event:   0.00561
  Hit rate:               47.722%
  Sharpe-like ratio:      -5.129

RF signal
  Events followed:        98,468
  Mean net return/event:  -0.00040
  Std net return/event:   0.00556
  Hit rate:               47.120%
  Sharpe-like ratio:      -5.539

XGB signal
  Events followed:        98,468
  Mean net return/event:  -0.00039
  Std net return/event:   0.00559
  Hit rate:               47.117%
  Sharpe-like ratio:      -5.304

LSTM signal
  Events followed:        98,468
  Mean net return/event:  -0.00039
  Std net return/event:   0.00557
  Hit rate:               47.061%
  Sharpe-like ratio:      -5.374


---
## Final Dashboard

A consolidated summary of every stage's key output: tick rows used and trading days spanned, the calibrated dollar-bar threshold and bar count, the fitted fractional-differentiation `d*`, CUSUM/triple-barrier event counts, the feature-selection result, purged-CV fold results, and OOS accuracy/F1 plus per-event return statistics for all four models side by side.

In [13]:
# ════════════════════════════════════════════════════════════════════════
# FINAL DASHBOARD
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print(" COMPLETE PIPELINE SUMMARY ".center(65, "="))
print("=" * 65)
rows = [
    ("DRY_RUN",              f"{DRY_RUN}  ({DRY_RUN_MODE})" if DRY_RUN else "False (full file)"),
    ("Tick rows used",       f"{len(TRADES):,}"),
    ("Trading days spanned", f"{n_days_span}"),
    ("Dollar bars",          f"{len(BARS):,}  (threshold={CFG['dollar_bar_usd']:,.0f})"),
    ("Min-d* (FFD)",         f"d* = {d_star:.4f}"),
    ("CUSUM events",         f"{len(t_events):,} -> {len(EVENTS):,} valid events"),
    ("Features",             f"{FEATURES.shape[1]} raw -> {len(top_feats)} selected"),
    ("CV folds completed",   f"{len(FOLD_METRICS)}"),
]
for name in model_names:
    rows.append((f"{name.upper()} OOS acc / f1",
                  f"{FOLD_METRICS[f'{name}_acc'].mean():.4f} / {FOLD_METRICS[f'{name}_f1'].mean():.4f}" if len(FOLD_METRICS) else "n/a"))
for name in model_names:
    s = STATS[name]
    rows.append((f"{name.upper()} hit rate / Sharpe-like", f"{s['hit_rate']:.3%} / {s['sharpe_like']:.3f}"))

width = max(len(r[0]) for r in rows) + 2
for name, val in rows:
    print(f"{name.ljust(width)}: {val}")


=================== COMPLETE PIPELINE SUMMARY ===================
DRY_RUN                        : False (full file)
Tick rows used                 : 10,702,026
Trading days spanned           : 4213
Dollar bars                    : 103,303  (threshold=3,002,353)
Min-d* (FFD)                   : d* = 0.2000
CUSUM events                   : 99,046 -> 98,999 valid events
Features                       : 25 raw -> 15 selected
CV folds completed             : 5
LOGREG OOS acc / f1            : 0.5117 / 0.4991
RF OOS acc / f1                : 0.5059 / 0.4581
XGB OOS acc / f1               : 0.5052 / 0.5162
LSTM OOS acc / f1              : 0.5043 / 0.5450
LOGREG hit rate / Sharpe-like  : 47.722% / -5.129
RF hit rate / Sharpe-like      : 47.120% / -5.539
XGB hit rate / Sharpe-like     : 47.117% / -5.304
LSTM hit rate / Sharpe-like    : 47.061% / -5.374
